# Perfilado de fuente: PortalInmobiliario

**Objetivo:** perfilar los 24 archivos mensuales de PortalInmobiliario (2023-2024) en
`data/raw/portal_inmobiliario/` para detectar problemas de calidad de datos antes de diseñar
la capa `staging`, y comparar el esquema con Toctoc (ver `01_exploration_toctoc.ipynb`).

Este notebook es solo exploratorio: no transforma ni guarda datos intermedios.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

RAW_DIR = Path("../data/raw/portal_inmobiliario")

## 0. Carga de los 24 archivos mensuales

In [2]:
files = sorted(RAW_DIR.glob("*/*.csv"))
print(f"Archivos encontrados: {len(files)}")
assert len(files) == 24, "Se esperaban 24 archivos mensuales (2023-01 a 2024-12)"

dfs = []
for f in files:
    d = pd.read_csv(f)
    d["archivo_origen"] = f.name
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
df.shape

Archivos encontrados: 24


(122273, 13)

## 1. Estructura general: columnas y tipos

In [3]:
df.dtypes

id                         str
fuente                     str
comuna                     str
region                     str
precio_clp               int64
precio_uf              float64
superficie_total_m2    float64
fecha_publicacion          str
fecha_scraping             str
tipo_propiedad             str
tipo_operacion             str
contact_type               str
archivo_origen             str
dtype: object

In [4]:
df.head(3)

,id,fuente,comuna,region,precio_clp,precio_uf,superficie_total_m2,fecha_publicacion,fecha_scraping,tipo_propiedad,tipo_operacion,contact_type,archivo_origen
0,PI-2023-01-002528,portalinmobiliario,Estación Central,Metropolitana,375000,10.71,38.7,2022-12-24 00:00:00.000000,2023-01-15 00:00:00.000000,Departamento,ARRIENDO,agency,2023-01.csv
1,PI-2023-01-000758,portalinmobiliario,San Joaquín,Metropolitana,465000,13.29,45.8,2022-12-22 00:00:00.000000,2023-01-01 00:00:00.000000,Departamento,arriendo,agency,2023-01.csv
2,PI-2023-01-005094,portalinmobiliario,Ñuñoa,Metropolitana,485000,13.86,35.5,2022-11-29 00:00:00.000000,2023-01-29 00:00:00.000000,DEPARTAMENTO,ARRIENDO,agency,2023-01.csv


## 2. Nulos por columna

In [5]:
nulos = df.isna().sum().to_frame("n_nulos")
nulos["pct_nulos"] = (nulos["n_nulos"] / len(df) * 100).round(2)
nulos.sort_values("n_nulos", ascending=False)

,n_nulos,pct_nulos
precio_uf,1480,1.21
comuna,659,0.54
id,0,0.00
region,0,0.00
fuente,0,0.00
precio_clp,0,0.00
superficie_total_m2,0,0.00
fecha_publicacion,0,0.00
fecha_scraping,0,0.00
tipo_propiedad,0,0.00


## 3. Duplicados

In [6]:
dup_exactos = df.duplicated().sum()
dup_id = df["id"].duplicated().sum()
print(f"Filas exactamente duplicadas (todas las columnas): {dup_exactos}")
print(f"Filas con id duplicado: {dup_id}")

Filas exactamente duplicadas (todas las columnas): 178
Filas con id duplicado: 178


In [7]:
ids_dup = df[df["id"].duplicated(keep=False)].sort_values("id")
ids_dup.groupby("id").nunique().max()

fuente                 1
comuna                 1
region                 1
precio_clp             1
precio_uf              1
superficie_total_m2    1
fecha_publicacion      1
fecha_scraping         1
tipo_propiedad         1
tipo_operacion         1
contact_type           1
archivo_origen         1
dtype: int64

## 4. Outliers en precio y superficie

In [8]:
df[["precio_clp", "precio_uf"]].describe()

,precio_clp,precio_uf
count,1.222730e+05,120793.000000
mean,7.746602e+05,21.533076
std,1.063716e+06,29.708064
min,2.500000e+05,6.760000
25%,4.900000e+05,13.650000
50%,6.050000e+05,16.860000
75%,8.100000e+05,22.430000
max,1.410350e+08,3811.760000


In [9]:
print("precio_clp <= 0:", (df["precio_clp"] <= 0).sum())
print("precio_uf <= 0:", (df["precio_uf"] <= 0).sum())
print("precio_uf nulo (precio_clp no nulo):",
      (df["precio_uf"].isna() & df["precio_clp"].notna()).sum())
print("superficie_total_m2 <= 0:", (df["superficie_total_m2"] <= 0).sum())
df["superficie_total_m2"].describe()

precio_clp <= 0: 0
precio_uf <= 0: 0
precio_uf nulo (precio_clp no nulo): 1480
superficie_total_m2 <= 0: 0


count    122273.000000
mean         57.612125
std          61.280752
min           1.000000
25%          39.100000
50%          49.100000
75%          63.700000
max        1200.000000
Name: superficie_total_m2, dtype: float64

In [10]:
# Candidatos a outlier: percentil 99.5 de precio_clp y precio_uf
for col in ["precio_clp", "precio_uf"]:
    p995 = df[col].quantile(0.995)
    print(f"{col}: p99.5 = {p995:.2f}, max = {df[col].max():.2f}, "
          f"n > p99.5 = {(df[col] > p995).sum()}")

precio_clp: p99.5 = 7058200.00, max = 141035000.00, n > p99.5 = 612
precio_uf: p99.5 = 196.62, max = 3811.76, n > p99.5 = 604


## 5. Fechas: parseabilidad y consistencia

In [11]:
fecha_publicacion = pd.to_datetime(df["fecha_publicacion"], errors="coerce")
fecha_scraping = pd.to_datetime(df["fecha_scraping"], errors="coerce")

print("fecha_publicacion sin parsear:", fecha_publicacion.isna().sum())
print("fecha_scraping sin parsear:", fecha_scraping.isna().sum())
print("fecha_publicacion rango:", fecha_publicacion.min(), "→", fecha_publicacion.max())
print("fecha_scraping rango:", fecha_scraping.min(), "→", fecha_scraping.max())
print("fecha_publicacion > fecha_scraping (inconsistente):", (fecha_publicacion > fecha_scraping).sum())

fecha_publicacion sin parsear: 0
fecha_scraping sin parsear: 0
fecha_publicacion rango: 2022-04-23 00:00:00 → 2024-12-28 00:00:00
fecha_scraping rango: 2023-01-01 00:00:00 → 2024-12-29 00:00:00
fecha_publicacion > fecha_scraping (inconsistente): 0


## 6. Consistencia de valores categóricos

In [12]:
for col in ["tipo_propiedad", "tipo_operacion", "contact_type", "region"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

--- tipo_propiedad ---
tipo_propiedad
Departamento    40899
DEPARTAMENTO    40748
departamento    40626
Name: count, dtype: int64

--- tipo_operacion ---
tipo_operacion
Arriendo    40851
ARRIENDO    40830
arriendo    40592
Name: count, dtype: int64

--- contact_type ---
contact_type
agency          79641
owner_direct    42632
Name: count, dtype: int64

--- region ---
region
Metropolitana    122273
Name: count, dtype: int64



In [13]:
comuna_no_nula = df["comuna"].dropna().astype(str)
con_espacios = sorted(c for c in comuna_no_nula.unique() if c != c.strip())
print(f"Comunas con espacios al inicio/final: {len(con_espacios)} de {comuna_no_nula.nunique()} valores únicos")
print(con_espacios)
print("Comunas únicas tras strip():", comuna_no_nula.str.strip().nunique())

Comunas con espacios al inicio/final: 50 de 125 valores únicos
[' Cerrillos', ' Conchalí', ' Estación Central', ' Huechuraba', ' Independencia', ' La Cisterna', ' La Florida', ' La Reina', ' Las Condes', ' Lo Barnechea', ' Macul', ' Maipú', ' Peñalolén', ' Providencia', ' Pudahuel', ' Puente Alto', ' Quilicura', ' Quinta Normal', ' Recoleta', ' Renca', ' San Joaquín', ' San Miguel', ' Santiago', ' Vitacura', ' Ñuñoa', 'Cerrillos ', 'Conchalí ', 'Estación Central ', 'Huechuraba ', 'Independencia ', 'La Cisterna ', 'La Florida ', 'La Reina ', 'Las Condes ', 'Lo Barnechea ', 'Macul ', 'Maipú ', 'Peñalolén ', 'Providencia ', 'Pudahuel ', 'Puente Alto ', 'Quilicura ', 'Quinta Normal ', 'Recoleta ', 'Renca ', 'San Joaquín ', 'San Miguel ', 'Santiago ', 'Vitacura ', 'Ñuñoa ']
Comunas únicas tras strip(): 75


## 7. Volumen de registros por archivo mensual

In [14]:
df["archivo_origen"].value_counts().sort_index()

archivo_origen
2023-01.csv    5268
2023-02.csv    4738
2023-03.csv    5388
2023-04.csv    5328
2023-05.csv    4890
2023-06.csv    5207
2023-07.csv    4861
2023-08.csv    5244
2023-09.csv    5069
2023-10.csv    4903
2023-11.csv    4754
2023-12.csv    5356
2024-01.csv    4888
2024-02.csv    5096
2024-03.csv    5047
2024-04.csv    4812
2024-05.csv    5224
2024-06.csv    5490
2024-07.csv    5206
2024-08.csv    4877
2024-09.csv    5140
2024-10.csv    4992
2024-11.csv    4987
2024-12.csv    5508
Name: count, dtype: int64

## 8. Comparación de esquema con Toctoc

In [15]:
toctoc_cols = {"id", "fuente", "comuna", "region", "precio", "divisa", "superficie_m2",
                "fecha_publicacion", "fecha_scraping", "tipo_propiedad", "tipo_operacion",
                "contact_type"}
portal_cols = set(df.columns) - {"archivo_origen"}

print("Solo en Toctoc:", toctoc_cols - portal_cols)
print("Solo en PortalInmobiliario:", portal_cols - toctoc_cols)
print("Columnas en común:", toctoc_cols & portal_cols)

Solo en Toctoc: {'superficie_m2', 'precio', 'divisa'}
Solo en PortalInmobiliario: {'precio_clp', 'superficie_total_m2', 'precio_uf'}
Columnas en común: {'fecha_publicacion', 'fuente', 'fecha_scraping', 'region', 'comuna', 'contact_type', 'tipo_propiedad', 'id', 'tipo_operacion'}


In [16]:
# ids de ambas fuentes no se solapan como texto (prefijos TT- / PI-),
# consistente con la regla de no asumir que `id` es único entre fuentes.
toctoc_ids = set()
for f in sorted(Path("../data/raw/toctoc").glob("*/*.csv")):
    toctoc_ids.update(pd.read_csv(f, usecols=["id"])["id"])

print("ids compartidos entre fuentes (por texto):", len(toctoc_ids & set(df["id"])))
print("prefijo típico Toctoc:", next(iter(toctoc_ids))[:3])
print("prefijo típico PortalInmobiliario:", df["id"].iloc[0][:3])

ids compartidos entre fuentes (por texto): 0
prefijo típico Toctoc: TT-
prefijo típico PortalInmobiliario: PI-


## Resumen de hallazgos — PortalInmobiliario

- **Esquema estable**: los 24 archivos comparten exactamente las mismas 12 columnas.
- **Nulos**: `precio_uf` tiene nulos aunque `precio_clp` esté presente (posible falla al
  convertir a UF en el scraping, no una fila sin precio).
- **Duplicados**: existen filas con `id` repetido, mismo patrón que en Toctoc.
- **Outliers**: hay valores de `precio_clp`/`precio_uf` muy por sobre el percentil 99.5 —
  candidatos a revisar antes de definir reglas de exclusión.
- **Casing inconsistente**: mismo problema que Toctoc en `tipo_propiedad`/`tipo_operacion`.
- **Espacios en `comuna`**: mismo problema que Toctoc.
- **Fechas**: parsean sin error y sin inconsistencias `fecha_publicacion > fecha_scraping`.

### Diferencias de esquema entre fuentes (clave para staging)

| Aspecto | Toctoc | PortalInmobiliario |
|---|---|---|
| Precio | 1 columna (`precio`) + `divisa` (UF o CLP mezclados) | 2 columnas ya separadas: `precio_clp`, `precio_uf` |
| Superficie | `superficie_m2` | `superficie_total_m2` |
| Casing `tipo_propiedad`/`tipo_operacion` | 3 variantes cada uno | 3 variantes cada uno |
| `contact_type` | tiene nulos | sin nulos |
| `id` | prefijo `TT-` | prefijo `PI-` |

No hay solape de `id` entre fuentes (confirmado arriba) — la integración en staging debe
tratar `id` como único solo dentro de cada fuente, nunca como clave global compartida.